# Preprocessing Data
## Introduction
The input data used to create the PyPSA-Canada-National model comes from a variety of sources, in a variety of formats that require pre-processing before being utilized in the model input scripts. This step must be run first, and the outputs must be saved in the ./results/processed_data folder. The original data sources are linked where possible.

In [ ]:
#Due to incompatibilities with linopy and xarray, pyarrow will be installed only for the purpose of building the model
%pip install pyarrow

In [2]:
import geopandas as gpd
import pandas as pd
import os

from helper_functions import CODERS_pull, download_and_unzip, mapLines, mapPoints, read_api_key

## Reading paths
Note: this script requires a CODERS API key. Create a text file called api_key.txt, write only your api key inside and save it to the data folder

In [3]:
## Paths
path = os.getcwd()
data_path = os.path.join(path, 'data')
results_path = os.path.join(path, 'results')
key = read_api_key(data_path)

### Variable Mapping Dictionaries

In [4]:
PRUID_map = {
    '10':'NL',
    '11':'PE',
    '12':'NS',
    '13':'NB',
    '24':'QC',
    '35':'ON',
    '46':'MB',
    '47':'SK',
    '48':'AB',
    '59':'BC',
    '60':'YT',
    '61':'NT',
    '62':'NU'
}

gen_type_map = {
    'Biomass': 'thermal',
    'MSW': 'thermal',
    'NG_CC': 'thermal',
    'NG_CG': 'thermal',
    'NG_SC': 'thermal',
    'biogas': 'thermal',
    'coal': 'thermal',
    'coal_CCS': 'thermal',
    'diesel_CT': 'thermal',
    'gasoline_CT': 'thermal',
    'hydro_daily': 'hydro',
    'hydro_monthly': 'hydro',
    'hydro_run': 'hydro',
    'nuclear': 'nuclear',
    'oil_CT': 'thermal',
    'oil_ST': 'thermal',
    'solar_PV': 'solar',
    'wind_ons': 'wind'
}

#### CODERS pulls
Skip this step if already pulled

In [5]:
node_data = CODERS_pull('nodes', key).set_index('node_code', drop=True)
generator_data = CODERS_pull('generators', key).set_index('generation_unit_name', drop=True)
storage_data = CODERS_pull('storage', key).set_index('storage_facility_name', drop=True)
line_data = CODERS_pull('transmission_lines', key).set_index('transmission_line_id', drop=True)
inter_lines = CODERS_pull('interjurisdictional_interties', key).set_index('transmission_circuit_id_starting', drop=True)

#Create CODERS folder if it doesn't exist
os.makedirs(os.path.join(data_path, 'CODERS'), exist_ok=True)

# Saving to CSV
node_data.to_csv(os.path.join(data_path, 'CODERS', 'nodes.csv'))
generator_data.to_csv(os.path.join(data_path, 'CODERS', 'generators.csv'))
storage_data.to_csv(os.path.join(data_path, 'CODERS', 'storage.csv'))
line_data.to_csv(os.path.join(data_path, 'CODERS', 'transmission_lines.csv'))
inter_lines.to_csv(os.path.join(data_path, 'CODERS', 'interjurisdictional.csv'))

### Reading data
The raw data files are read in this section. The province map is from the __[Statistics Canada Census Boundary Areas](https://www12.statcan.gc.ca/census-recensement/alternative_alternatif.cfm?l=eng&dispext=zip&teng=lcd_000b21a_e.zip&k=%20%20%20136594&loc=//www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/files-fichiers/lcd_000b21a_e.zip)__, the node data, generators, and lines are all taken from __[CODERS](https://coders.cme-emh.ca/login)__, and finally the population data is taken from the __[Statistics Canada 2021 Census of Population (Census Divisions (CDs))](https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/download-telecharger.cfm?Lang=E)__. For the census data, only the ALT_GEO_CODE, and C1_COUNT_TOTAL columns are used, the CHARACTERISTIC_NAME column is filtered for only "Population, 2021" and the filtered results are copied to a new csv file called census_population.csv in the data folder.

In [6]:
## Download files
os.makedirs(os.path.join(data_path, 'map_files'), exist_ok=True)
census_map_link = "https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/files-fichiers/lcd_000b21a_e.zip"
census_filepath = os.path.join(data_path, 'map_files', 'census_map.zip')
census_outpath = os.path.join(data_path, 'map_files', 'census_map')
download_and_unzip(census_map_link, census_filepath, census_outpath)

census_population_link = "https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/download-telecharger/comp/GetFile.cfm?Lang=E&FILETYPE=CSV&GEONO=004"
census_population_filepath = os.path.join(data_path, 'census_population.zip')
census_population_outpath = os.path.join(data_path, 'census_population')
download_and_unzip(census_population_link, census_population_filepath, census_population_outpath)

## Reading files
province_map = gpd.read_file(os.path.join(data_path, 'map_files', 'census_map', 'lcd_000b21a_e.shp')).rename(columns={'CDNAME':'CSDNAME'})
# population_old = pd.read_csv(os.path.join(data_path, 'census_population.csv'), index_col=0)['C1_COUNT_TOTAL']
population = pd.read_csv(os.path.join(data_path, 'census_population', r'98-401-X2021004_English_CSV_data.csv'), encoding='latin-1', index_col=2)
population = population[['GEO_NAME', 'CHARACTERISTIC_NAME', 'C1_COUNT_TOTAL']]
population = population[population.CHARACTERISTIC_NAME == 'Population, 2021']['C1_COUNT_TOTAL']
# CODERS
node_data = pd.read_csv(os.path.join(data_path, 'CODERS', 'nodes.csv'), index_col=0)
generator_data = pd.read_csv(os.path.join(data_path, 'CODERS', 'generators.csv'), index_col=0)
line_data = pd.read_csv(os.path.join(data_path, 'CODERS', 'transmission_lines.csv'), index_col=0)
inter_lines = pd.read_csv(os.path.join(data_path, 'CODERS', 'interjurisdictional.csv'), index_col=0)

Download complete
Extracting c:\Users\ndematos\Desktop\pypsa_canada\PyPSA-Canada-National\data\map_files\census_map.zip...
Extracted to c:\Users\ndematos\Desktop\pypsa_canada\PyPSA-Canada-National\data\map_files\census_map/
Download complete
Extracting c:\Users\ndematos\Desktop\pypsa_canada\PyPSA-Canada-National\data\census_population.zip...
Extracted to c:\Users\ndematos\Desktop\pypsa_canada\PyPSA-Canada-National\data\census_population/


### Formatting Data
#### Map data

In [7]:
province_map = province_map.to_crs('EPSG: 4326')
province_map = province_map.set_index('CDUID', drop=True)
province_map.index = province_map.index.astype(int)
province_map['province'] = province_map.PRUID.astype(str).str[:2].map(PRUID_map) # identifying province based on 2 digit code

#### Node Data

In [8]:
map_nodes = mapPoints(node_data[['latitude', 'longitude']]) # creating geodataframe
# Finding which census area each bus falls within
node_lookup = gpd.sjoin(map_nodes, province_map[['geometry', 'province']], how='left', predicate='within')
# Nodes which do not fall in any area - bugs
bugged_nodes = node_lookup[node_lookup.CDUID.isna()]
# ignore international transfers
node_lookup = node_lookup.drop(bugged_nodes[bugged_nodes.index.str.contains('US')].index)
bugged_nodes = bugged_nodes[~bugged_nodes.index.str.contains('US')]
print('-----BUGGED NODES-----')
print(bugged_nodes)

c:\Users\ndematos\envs\pypsa_canada_py312\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


-----BUGGED NODES-----
             latitude  longitude                    geometry  CDUID province
node_code                                                                   
NL_HOL_GSS  47.450743 -53.127667  POINT (-53.12767 47.45074)    NaN      NaN


##### Fixing "bugged" nodes
Manually enter a dict containing the CODERS node codes in the above bugged node list along with their associated CDUID/province and geometry.

In [9]:
bugged_nodes = {
    'NL_HOL_GSS': {'CDUID': 1010, 'province':'NL', 'geometry':map_nodes.at['NL_HOL_GSS', 'geometry']}
}

for row, columns in bugged_nodes.items():
    for col, val in columns.items():
        node_lookup.at[row, col] = val

node_lookup['CDUID'] = node_lookup['CDUID'].astype(int)
node_lookup = node_lookup[['CDUID', 'province', 'geometry']]

In [10]:
province_map#[province_map['CSDNAME'] == 'Division No.1']

,DGUID,CSDNAME,CDTYPE,LANDAREA,PRUID,geometry,province
CDUID,,,,,,,
1001,2021A00031001,Division No. 1,CDR,9104.5799,10,"MULTIPOLYGON (((-53.44465 46.67495, -53.44474 ...",NL
1002,2021A00031002,Division No. 2,CDR,5915.5695,10,"MULTIPOLYGON (((-54.22551 47.52866, -54.22567 ...",NL
1003,2021A00031003,Division No. 3,CDR,19272.1069,10,"MULTIPOLYGON (((-57.6093 47.62051, -57.60937 4...",NL
1004,2021A00031004,Division No. 4,CDR,7019.9723,10,"MULTIPOLYGON (((-59.29168 47.97929, -59.29182 ...",NL
1005,2021A00031005,Division No. 5,CDR,10293.7618,10,"MULTIPOLYGON (((-58.32368 49.25135, -58.32372 ...",NL
...,...,...,...,...,...,...,...
6105,2021A00036105,Region 5,REG,152135.9223,61,"POLYGON ((-110.04698 62.91817, -109.67964 62.8...",NT
6106,2021A00036106,Region 6,REG,182201.5022,61,"POLYGON ((-114.31376 66.05472, -112.58339 65.4...",NT
6204,2021A00036204,Qikiqtaaluk,REG,970554.6112,62,"MULTIPOLYGON (((-80.34393 51.41927, -80.34391 ...",NU


#### Generator data

In [11]:
## Generator data
generators = generator_data[['network_node_code', 'gen_type', 'unit_installed_capacity']]
generators['gen_type'] = generators.gen_type.map(gen_type_map)
generators = pd.merge(generators, node_lookup['CDUID'], how='left', left_on='network_node_code', right_index=True)
generation = generators.groupby(['CDUID', 'gen_type']).sum()['unit_installed_capacity'].reset_index()
generation = generation.pivot(index='CDUID', columns='gen_type', values='unit_installed_capacity').fillna(0).add_suffix('_capacity')
generation['total_capacity'] = generation.sum(axis=1)

C:\Users\ndematos\AppData\Local\Temp\ipykernel_34180\3960095936.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  generators['gen_type'] = generators.gen_type.map(gen_type_map)


#### Line data

In [12]:
inter_lines = inter_lines[inter_lines.to_country == 'Canada'] # exclude international
inter_lines = inter_lines.rename(columns={'line_length_km': 'line_segment_length_km'})
all_lines = pd.concat([line_data, inter_lines])
all_lines = all_lines[['network_node_code_starting', 'network_node_code_ending', 'line_segment_length_km', 'voltage', 'ttc_summer']]
all_lines = pd.merge(all_lines, node_lookup, how='left', left_on='network_node_code_starting', right_index=True)
all_lines = pd.merge(all_lines, node_lookup, how='left', left_on='network_node_code_ending', right_index=True, suffixes=['_1', '_2'])
all_lines = all_lines[~(all_lines.network_node_code_starting.str.contains('_INT') | all_lines.network_node_code_ending.str.contains('_INT'))]
bugged_lines = all_lines[(all_lines.CDUID_1.isna() | all_lines.CDUID_2.isna())].copy()
print('-----BUGGED LINES-----')
bugged_lines = bugged_lines[(~bugged_lines.network_node_code_starting.str.contains('_INT')) | (~bugged_lines.network_node_code_ending.str.contains('_INT'))]
print(bugged_lines)

-----BUGGED LINES-----
Empty DataFrame
Columns: [network_node_code_starting, network_node_code_ending, line_segment_length_km, voltage, ttc_summer, CDUID_1, province_1, geometry_1, CDUID_2, province_2, geometry_2]
Index: []


In [13]:
all_lines = all_lines[(all_lines.CDUID_1.notna()) & (all_lines.CDUID_2.notna())]
# required because of bugged nodes outside of census areas, merge changes CDUID to a float
all_lines['CDUID_1'] = all_lines['CDUID_1'].astype(int) 
all_lines['CDUID_2'] = all_lines['CDUID_2'].astype(int)
map_lines = mapLines(all_lines)
map_lines = map_lines.rename(columns={'ttc_summer':'capacity', 'network_node_code_starting':'node_1', 'network_node_code_ending':'node_2'}).drop(['geometry_1', 'geometry_2'], axis=1)

c:\Users\ndematos\envs\pypsa_canada_py312\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


#### Misc
Finding any regions not connected by transmission infrastructure and merging the population to the province map.

In [14]:
## Exclude disconnected regions (may be bugged or real)
connected_regions = set(all_lines.CDUID_1).union(set(all_lines.CDUID_2))
province_map['Connected'] = province_map.index.isin(connected_regions)
print('-----DISCONNECTED REGIONS-----')
print(province_map[~province_map['Connected']][['CSDNAME', 'province']])

### CREATING ZONE DATA
province_map['population'] = population
province_map = pd.merge(province_map, generation, how='left', left_index=True, right_index=True)

-----DISCONNECTED REGIONS-----
                                            CSDNAME province
CDUID                                                       
1011                                Division No. 11       NL
2401   Communauté maritime des Îles-de-la-Madeleine       QC
2417                                        L'Islet       QC
2420                                L'Île-d'Orléans       QC
5945                                  Central Coast       BC
5957                                        Stikine       BC
6001                                          Yukon       YT
6101                                       Region 1       NT
6102                                       Region 2       NT
6103                                       Region 3       NT
6104                                       Region 4       NT
6105                                       Region 5       NT
6106                                       Region 6       NT
6204                                    Qikiqtaaluk   

### Saving Data

In [15]:
## Geopandas
os.makedirs(os.path.join(data_path, 'processed_data'), exist_ok=True)
gpd.GeoDataFrame(map_lines).reset_index(drop=True).to_feather(os.path.join(data_path, 'processed_data', 'line_data.feather'))
gpd.GeoDataFrame(node_lookup).to_feather(os.path.join(data_path, 'processed_data', 'node_data.feather'))
province_map.to_feather(os.path.join(data_path, 'processed_data', 'zone_data.feather'))

## Human readable
os.makedirs(os.path.join(results_path, 'intermediary_outputs'), exist_ok=True)
province_map.drop('geometry', axis=1).to_csv(os.path.join(results_path, 'intermediary_outputs', 'zone_data.csv'))
map_lines.drop('geometry',axis=1).to_csv(os.path.join(results_path, 'intermediary_outputs', 'line_data.csv'))